In [2]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

server = r"LAPTOP-71G1NMQR\SQLEXPRESS"
database = "AQI_Analytics"

connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(connection_string)
)

print("SQLAlchemy engine created successfully!")

SQLAlchemy engine created successfully!


SQL Server connection actually test

In [3]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("SELECT DB_NAME() AS database_name"))
    print(result.fetchone())

('AQI_Analytics',)


In [4]:
import pandas as pd

df = pd.read_csv("aqi_clean.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (54000, 20)

Columns:
['time', 'pm2_5', 'pm10', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'us_aqi', 'city', 'state', 'latitude', 'longitude', 'date', 'year', 'month', 'day', 'hour', 'day_of_week', 'day_name', 'aqi_category']


In [5]:
df.to_sql(
    name="AQI_Data",
    con=engine,
    schema="dbo",
    if_exists="replace",
    index=False,
    chunksize=5000
)

print("AQI_Data table uploaded successfully!")

AQI_Data table uploaded successfully!


In [6]:
##verify data from sql serever
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT COUNT(*) AS total_rows FROM dbo.AQI_Data")
    )
    
    print("Total rows in SQL Server:", result.fetchone()[0])

Total rows in SQL Server: 54000
